# Semaine 2 — Jour 4 : Conversation State

Notebook étudiant généré à partir du Markdown source.

## Objectifs

- Comprendre pourquoi un agent multi-tours a besoin d’un état explicite.
- Distinguer historique, state et memory.
- Collecter progressivement des slots métier.
- Détecter les champs manquants.
- Sérialiser et restaurer un état conversationnel.
- Tester l’isolation entre sessions.

## Modèle mental

```text
User message → Load state → Extract intent/slots → Update state → Missing fields → Response/action → Save state
```

## Extrait du chapitre

# Chapitre — Conversation State

## 1. Le problème d’un agent stateless

Un agent stateless traite chaque message comme s’il était indépendant.

Exemple :

```text
Utilisateur : Je veux un remboursement.
Assistant : Quel est votre numéro de commande ?
Utilisateur : ORD-1001
```

Un système stateless voit seulement le dernier message :

```text
ORD-1001
```

Il ne sait plus :

- que l’utilisateur demande un remboursement ;
- pourquoi le numéro de commande est fourni ;
- quel champ vient d’être complété ;
- quelles informations restent à demander ;
- quelle étape du processus est en cours.

Pour un chatbot de démonstration, cela peut parfois passer.

Pour un agent backend, c’est un problème d’architecture.

Un agent professionnel doit porter une représentation explicite de la conversation.

## 2. Définition

Le **Conversation State** est l’état applicatif courant d’une conversation.

Il représente ce que le backend sait actuellement du dialogue.

Il peut contenir :

- l’identifiant de session ;
- l’identifiant utilisateur ;
- l’historique récent ;
- l’intention courante ;
- les champs métier déjà collectés ;
- les champs encore manquants ;
- les résultats d’outils ;
- le statut de progression ;
- les métadonnées utiles au contrôle du flux.

Exemple simplifié :

```json
{
  "session_id": "session_001",
  "user_id": "user_123",
  "intent": "refund",
  "slots": {
    "order_id": "ORD-1001",
    "email": "lea@example.com"
  },
  "missing_fields": ["reason"],
  "status": "collecting",
  "turn_count": 3
}
```

Cet objet permet à l’application de prendre une décision stable :

```text
Il manque encore la raison du remboursement.
La prochaine réponse doit demander cette information.
```

## 3. Conversation State, historique et mémoire

Il faut distinguer trois notions proches.

### Historique conversationnel

L’historique est la liste des messages échangés.

Exemple :

```json
[
  {"role": "user", "content": "Je veux un remboursement."},
  {"role": "assistant", "content": "Quel est votre numéro de commande ?"},
  {"role": "user", "content": "ORD-1001"}
]
```

L’historique est utile pour reconstruire le contexte linguistique.

Mais il n’est pas suffisant pour piloter un backend.

Un backend ne veut pas reparcourir tout le texte à chaque tour pour deviner l’état métier.

### Conversation State

Le state est une représentation structurée de la situation courante.

Il répond à des questions comme :

- quelle est l’intention active ?
- quels champs sont remplis ?
- quels champs manquent ?
- quelle étape du workflow est en cours ?
- une action peut-elle être déclenchée ?

Le state doit être lisible par du code.

### Memory

La mémoire, qui sera traitée au jour suivant, concerne la conservation d’informations au-delà de la conversation immédiate.

Exemples :

- préférence de langue d’un utilisateur ;
- historique long terme d’un client ;
- résumé durable d’une relation ;
- faits personnalisés persistants.

La différence principale :

```text
Conversation State = état opérationnel de la conversation courante.
Memory = connaissances persistantes utilisables dans plusieurs conversations.
```

## 4. Pourquoi le state doit être explicite

On pourrait demander au modèle :

```text
Relis la conversation et décide quoi faire.
```

Cette approche est fragile.

Elle dépend :

- de la fenêtre de contexte ;
- de la qualité du prompt ;
- de la capacité du modèle à ne pas oublier ;
- de la cohérence des messages précédents ;
- du coût en tokens ;
- du comportement non déterministe du modèle.

Un state explicite déplace une partie du contrôle vers le backend.

Le modèle peut aider à extraire ou interpréter.

Mais l’application conserve le contrat de progression.

## 5. Les composants d’un état conversationnel

Un bon state minimal contient plusieurs couches.

### Identité de session

La session évite de mélanger les conversations.

```json
{
  "session_id": "support_2026_0001",
  "user_id": "user_123"
}
```

L’identifiant de session ne doit pas être confondu avec l’identifiant utilisateur.

Un même utilisateur peut avoir plusieurs sessions.

Une même session doit appartenir à un seul utilisateur.

### Historique court

L’historique court peut être utile pour :

- afficher la conversation ;
- donner du contexte au modèle ;
- auditer un comportement ;
- comprendre pourquoi un état a changé.

Il faut toutefois éviter d’y stocker trop de données.

### Intention active

L’intention représente le besoin principal détecté.

Exemples :

```text
refund
order_status
technical_issue
unknown
```

L’intention peut être inconnue au début.

Elle peut être révisée si l’utilisateur clarifie sa demande.

Mais une révision doit être explicite.

### Slots métier

Les slots sont les champs nécessaires au traitement.

Exemple pour une demande de remboursement :

```json
{
  "order_id": "ORD-1001",
  "email": "lea@example.com",
  "reason": "double_billing"
}
```

Les slots sont essentiels parce qu’ils permettent de savoir si une action peut être exécutée.

### Champs manquants

Les champs manquants peuvent être calculés.

Exemple :

```text
required_fields(refund) = order_id, email, reason
known_slots = order_id, email
missing_fields = reason
```

Il est préférable de recalculer les champs manquants à partir de l’intention et des slots plutôt que de les maintenir manuellement.

### Résultats d’outils

Un agent peut appeler des outils.

Les résultats d’outils peuvent enrichir le state :

```json
{
  "tool_results": {
    "get_order_status": {
      "status": "delivered",
      "delivered_at": "2026-08-01"
    }
  }
}
```

Il faut distinguer :

- ce que l’utilisateur a dit ;
- ce qu’un outil a confirmé ;
- ce que le modèle a inféré.

### Statut

Le statut indique la progression.

Exemples :

```text
collecting
ready_for_action
completed
blocked
```

Le statut permet d’éviter d’appeler un outil trop tôt.

## 6. Cycle de vie du state

Un état conversationnel suit un cycle.

```mermaid
stateDiagram-v2
    [*] --> New
    New --> Collecting: premier message
    Collecting --> Collecting: information partielle
    Collecting --> ReadyForAction: champs requis complets
    ReadyForAction --> Completed: action exécutée
    ReadyForAction --> Blocked: validation échouée
    Blocked --> Collecting: correction utilisateur
    Completed --> [*]
```

À chaque tour :

1. charger l’état ;
2. ajouter le message utilisateur ;
3. extraire les informations utiles ;
4. mettre à jour les slots ;
5. recalculer les champs manquants ;
6. décider la prochaine action ;
7. ajouter la réponse assistant ;
8. sauvegarder l’état.

## 7. State update

La mise à jour d’état est une opération critique.

Elle doit être :

- déterministe autant que possible ;
- testable ;
- observable ;
- réversible si nécessaire ;
- stricte sur les champs acceptés.

Exemple de pseudo-code :

```python
def handle_turn(state, user_message):
    state.messages.append(user_message)

    extraction = extract_information(user_message, state)

    state.intent = resolve_intent(state.intent, extraction.intent)
    state.slots.update(extraction.slots)

    missing = missing_required_fields(state)

    if missing:
        assistant_message = ask_for(missing[0])
        state.status = "collecting"
    else:
        assistant_message = confirm_ready(state)
        state.status = "ready_for_action"

    state.messages.append(assistant_message)
    return state
```

Le modèle peut intervenir dans `extract_information`.

Mais la logique de mise à jour doit rester sous contrôle applicatif.

## 8. Gestion des corrections utilisateur

Un utilisateur peut corriger une information :

```text
Utilisateur : Ma commande est ORD-1001.
Assistant : Quelle adresse email ?
Utilisateur : Pardon, la commande est ORD-2002.
```

Le state doit permettre une correction.

Mais une correction peut avoir des impacts.

Si un outil a déjà été appelé avec `ORD-1001`, le résultat de cet outil devient potentiellement invalide.

Le système doit donc décider :

- remplacer le slot ;
- invalider certains résultats d’outils ;
- journaliser la correction ;
- redemander confirmation si le changement est critique.

## 9. Isolation des sessions

Un problème grave dans les agents stateful est la fuite d’état.

Exemple dangereux :

```text
Session A : email = alice@example.com
Session B : l’agent réutilise accidentellement alice@example.com
```

Cette erreur peut créer :

- une fuite de données personnelles ;
- une mauvaise action métier ;
- une violation de conformité ;
- une perte de confiance utilisateur.

Règles de base :

- ne jamais utiliser une variable globale mutable pour stocker l’état d’un utilisateur ;
- toujours indexer l’état par session ;
- vérifier que `session_id` et `user_id` correspondent ;
- tester explicitement deux sessions parallèles ;
- effacer ou expirer les états inactifs.

## 10. Sérialisation

Un state doit souvent être sauvegardé.

La sérialisation JSON est un bon format pédagogique.

Exemple :

```python
serialized = json.dumps(state.to_dict())
restored = ConversationState.from_dict(json.loads(serialized))
```

En production, l’état peut être stocké dans :

- Redis ;
- PostgreSQL ;
- un store managé ;
- un système de session du framework agentique ;
- un backend conversationnel fournisseur.

Le choix dépend :

- de la durée de vie de la session ;
- des exigences de confidentialité ;
- du volume de conversations ;
- des besoins d’audit ;
- du coût d’accès ;
- de la stratégie de rétention.

## 11. Relation avec les APIs modernes

Les plateformes agentiques proposent souvent des mécanismes de continuité de conversation.

Ces mécanismes peuvent gérer l’historique ou une partie de l’état.

Mais cela ne supprime pas le besoin d’un state applicatif métier.

Un backend doit encore savoir :

- quelle intention métier est active ;
- quels champs requis sont validés ;
- quel outil peut être appelé ;
- quelle action a déjà été exécutée ;
- quelles données doivent être stockées ou supprimées.

Le state applicatif est donc complémentaire au state géré par un fournisseur ou un SDK.

## 12. Design recommandé

Pour un premier agent de production, utiliser une structure simple :

```text
ConversationState
├── session_id
├── user_id
├── intent
├── slots
├── tool_results
├── messages
├── turn_count
└── status
```

Puis ajouter des couches seulement quand elles sont nécessaires :

- version de schéma ;
- deadline ou expiration ;
- consentements ;
- résumé compressé ;
- traces d’audit ;
- provenance des champs ;
- stratégie de nettoyage.

Ne pas commencer par un state trop complexe.

Un état trop riche devient difficile à maintenir.

## 13. Anti-patterns

### Anti-pattern 1 — Tout stocker dans le prompt

Le prompt ne remplace pas le state.

Il peut contenir un résumé de l’état, mais il ne doit pas être la source de vérité.

### Anti-pattern 2 — Tout stocker dans l’historique

L’historique est du texte.

Le backend a besoin de données structurées.

### Anti-pattern 3 — Corriger silencieusement

Si l’utilisateur corrige une donnée, le système doit gérer explicitement le changement.

### Anti-pattern 4 — Utiliser un state global

Un dictionnaire global peut être acceptable pour un exercice local.

En production, il faut un store isolé, sécurisé et expirant.

### Anti-pattern 5 — Confondre state et mémoire

Le state répond à la question :

```text
Où en est cette conversation ?
```

La mémoire répond à la question :

```text
Que sait-on durablement de cet utilisateur ou de ce domaine ?
```

## 14. Synthèse

Le Conversation State est la colonne vertébrale d’un agent multi-tours.

Il transforme un dialogue libre en processus contrôlable.

Un agent robuste ne s’appuie pas uniquement sur le modèle pour se souvenir.

Il maintient un état applicatif explicite, validé, sérialisable et testable.

Le jour suivant étendra cette logique vers la **Memory** : ce qui doit survivre au-delà de la conversation courante.

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd() / "book" / "week02" / "day04" / "labs",
    Path.cwd().parent / "book" / "week02" / "day04" / "labs",
    Path("/mnt/data/ai-engineering-bootcamp/book/week02/day04/labs"),
]

for lab_path in candidates:
    if lab_path.exists():
        sys.path.append(str(lab_path))
        break

from conversation_state_agent import (
    execute_ready_action,
    handle_user_message,
    initialize_state,
    missing_required_fields,
    restore_state,
    serialize_state,
)

print(f"Lab path: {lab_path}")

In [ ]:
state = initialize_state("notebook_session", "learner_001")

for message in [
    "Je veux un remboursement.",
    "ORD-1001",
    "lea@example.com",
    "J'ai été facturée deux fois.",
]:
    state, reply = handle_user_message(state, message)
    print("Utilisateur:", message)
    print("Assistant:  ", reply)
    print("State:      ", state.intent, state.slots, state.status)
    print("---")

In [ ]:
payload = serialize_state(state)
restored = restore_state(payload)

print(payload)
print(restored.to_dict() == state.to_dict())

In [ ]:
result = execute_ready_action(restored)
result

## Exercices

# Exercices — Conversation State

## Exercice 1 — Identifier le problème de statelessness

Analyse la conversation suivante :

```text
Utilisateur : Je veux suivre ma commande.
Assistant : Quel est votre numéro de commande ?
Utilisateur : ORD-1001
```

Explique pourquoi le dernier message seul ne suffit pas pour générer une bonne réponse.

Ta réponse doit mentionner :

- l’intention ;
- le champ collecté ;
- les champs potentiellement manquants ;
- la prochaine action.

## Exercice 2 — Concevoir un state minimal

Conçois un objet `ConversationState` pour un agent de support.

Il doit contenir au minimum :

- `session_id` ;
- `user_id` ;
- `intent` ;
- `slots` ;
- `messages` ;
- `tool_results` ;
- `turn_count` ;
- `status`.

Pour chaque champ, explique :

- son rôle ;
- son type probable ;
- pourquoi il est utile au backend.

## Exercice 3 — Calculer les champs manquants

On définit les champs requis suivants :

```python
REQUIRED_FIELDS = {
    "order_status": {"order_id", "email"},
    "refund": {"order_id", "email", "reason"},
    "technical_issue": {"email", "issue_summary"},
}
```

État courant :

```json
{
  "intent": "refund",
  "slots": {
    "order_id": "ORD-1001"
  }
}
```

Quels champs manquent ?

Quelle question l’assistant doit-il poser ensuite ?

## Exercice 4 — Détecter une fuite d’état

Deux conversations existent.

Session A :

```json
{
  "session_id": "A",
  "user_id": "alice",
  "slots": {
    "email": "alice@example.com"
  }
}
```

Session B :

```json
{
  "session_id": "B",
  "user_id": "bob",
  "slots": {}
}
```

Un bug provoque l’utilisation de l’email d’Alice dans la session de Bob.

Explique :

- pourquoi ce bug est grave ;
- quelle règle d’architecture il viole ;
- quel test devrait exister ;
- quelle solution de stockage éviterait ce risque.

## Exercice 5 — State ou Memory ?

Classe les informations suivantes dans `Conversation State`, `Memory` ou `Ni l’un ni l’autre`.

1. Le numéro de commande donné dans la conversation en cours.
2. La préférence durable de l’utilisateur pour des réponses en français.
3. Un mot de passe envoyé par erreur dans le chat.
4. Le statut `ready_for_action`.
5. Le résumé long terme des projets préférés d’un utilisateur.
6. Le dernier message assistant.
7. Le résultat d’un outil appelé dans la conversation actuelle.
8. Une donnée personnelle non nécessaire à la tâche.

## Exercice 6 — Sérialisation

Explique pourquoi un état conversationnel doit pouvoir être sérialisé.

Donne deux exemples de stores possibles.

Explique aussi pourquoi la sérialisation ne suffit pas à garantir la sécurité.

## Challenge

# Challenge — Agent de support stateful

## Objectif

Construire un agent de support client capable de maintenir un état conversationnel sur plusieurs tours.

L’agent doit gérer au moins trois intentions :

- suivi de commande ;
- remboursement ;
- problème technique.

## Contraintes fonctionnelles

### Intention `order_status`

Champs requis :

- `order_id` ;
- `email`.

### Intention `refund`

Champs requis :

- `order_id` ;
- `email` ;
- `reason`.

### Intention `technical_issue`

Champs requis :

- `email` ;
- `issue_summary`.

## Comportement attendu

L’agent doit :

1. initialiser un état par session ;
2. détecter ou conserver l’intention active ;
3. extraire les champs disponibles dans les messages ;
4. calculer les champs manquants ;
5. poser une seule question ciblée à la fois ;
6. passer à `ready_for_action` lorsque tous les champs requis sont présents ;
7. refuser d’exécuter une action si l’état est incomplet ;
8. sérialiser et restaurer l’état ;
9. empêcher la fuite d’état entre sessions.

## Exemple attendu

```text
Utilisateur : Je veux un remboursement.
Assistant : Quel est votre numéro de commande ?
Utilisateur : ORD-1001
Assistant : Quelle adresse email est associée à la demande ?
Utilisateur : lea@example.com
Assistant : Quelle est la raison de votre demande de remboursement ?
Utilisateur : J’ai été facturée deux fois.
Assistant : Merci, j’ai les informations nécessaires pour traiter la demande de remboursement.
```

## Livrables attendus

- une classe ou dataclass `ConversationState` ;
- une fonction `handle_user_message` ;
- une fonction `missing_required_fields` ;
- une fonction de sérialisation ;
- une fonction de restauration ;
- au moins six tests ;
- un court commentaire expliquant la différence entre state et memory.

## Critères de validation

Le challenge est réussi si :

- le comportement multi-tours fonctionne ;
- les sessions sont isolées ;
- les champs manquants sont corrects ;
- l’action n’est jamais déclenchée trop tôt ;
- l’état restauré est équivalent à l’état sauvegardé ;
- le code reste compréhensible et exécutable sans service externe.

## Références

# Références — Conversation State

## Références principales

- OpenAI Agents SDK — Sessions.
- OpenAI Agents SDK — Running agents.
- OpenAI Agents SDK — Agents.
- OpenAI Platform — Responses API.
- OpenAI Platform — Function calling.
- OpenAI Platform — Structured Outputs.

## Concepts à revoir

- Conversation history.
- Session.
- Slot filling.
- State machine.
- Serialization.
- Idempotency.
- Session isolation.
- Personally identifiable information.
- Short-term state vs long-term memory.

## À retenir

Un framework ou une API peut aider à gérer l’historique de conversation.

Mais l’état métier d’un agent reste une responsabilité applicative.

Pour un AI Backend Engineer, le point clé est de concevoir un state :

- explicite ;
- minimal ;
- validable ;
- isolé par session ;
- sérialisable ;
- observable ;
- sécurisé.